In [1]:
# =============================================================
# CARGA DEL MODELO — Carga modelo_barrera1.pth ya entrenado con 3 clases
# No requiere reentrenamiento. Listo en ~30 segundos.
# Modelo guardado: 2026-08-13 22:29 (4330 KB)
# =============================================================

import warnings
warnings.filterwarnings('ignore')
import torch, torch.nn as nn, gc
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score,
    roc_curve, auc
)

BASE_DIR    = Path('../../').resolve()
RUTA_MODELO = BASE_DIR / 'modelo' / 'modelo_barrera1.pth'

# 3 clases requeridas por el supervisor
CLASES     = {'actas_nacimiento': 0, 'curp': 1, 'otros': 2}
CLASES_INV = {v: k for k, v in CLASES.items()}

dispositivo = torch.device('cpu')
print(f'Dispositivo: {dispositivo}')

tf_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class DocumentDataset(torch.utils.data.Dataset):
    def __init__(self, rutas, etiquetas, transform):
        self.rutas, self.etiquetas, self.transform = rutas, etiquetas, transform
    def __len__(self): return len(self.rutas)
    def __getitem__(self, idx):
        img = Image.open(self.rutas[idx]).convert('RGB')
        return self.transform(img), torch.tensor(self.etiquetas[idx], dtype=torch.long)

# Cargar dataset (solo actas y curp — otros tiene muy pocas imagenes de prueba)
df   = pd.read_csv(BASE_DIR / 'manifiesto.csv')
df_v = df[df['clase'].isin(CLASES) & (df['es_aumentada'] == False)].copy()
df_v = df_v[df_v['ruta_imagen'].apply(lambda r: (BASE_DIR / r).exists())]
X    = [str(BASE_DIR / r) for r in df_v['ruta_imagen']]
Y    = [CLASES[c] for c in df_v['clase']]

_, X_test, _, Y_test = train_test_split(X, Y, test_size=0.20, random_state=42, stratify=Y)
loader_test = DataLoader(DocumentDataset(X_test, Y_test, tf_test), batch_size=8, shuffle=False)
print(f'Imagenes de prueba: {len(X_test)}')

# Cargar modelo con 3 clases
modelo = models.mobilenet_v3_small(weights=None)
modelo.classifier = nn.Sequential(
    nn.Linear(576, 256), nn.Hardswish(), nn.Dropout(p=0.3), nn.Linear(256, 3)
)
modelo.load_state_dict(torch.load(RUTA_MODELO, map_location=dispositivo))
modelo = modelo.to(dispositivo)
modelo.eval()
print(f'Modelo cargado: {RUTA_MODELO.name}')
print('Listo para calcular metricas!')


Dispositivo: cpu
Imagenes de prueba: 247
Modelo cargado: modelo_barrera1.pth
Listo para calcular metricas!


# SISTEMA DE VERIFICACIÓN AUTOMÁTICA DE DOCUMENTOS ESCOLARES
## Pipeline de Evaluación y Métricas Científicas
---
Implementación de evaluación sistemática para el modelo de clasificación documental (Actas de Nacimiento, CURP y Otros).


In [2]:
# =============================================================
# SISTEMA DE VERIFICACIÓN AUTOMÁTICA DE DOCUMENTOS ESCOLARES
# Pipeline de 5 etapas para Actas de Nacimiento y CURPs
# Referencia: Bulatov et al. (2021). MIDV-2020. arXiv:2107.00396
# =============================================================

import warnings                  # Supresión de advertencias no críticas
warnings.filterwarnings('ignore')

import pandas as pd              # Manipulación y análisis de datos tabulares
from pathlib import Path         # Gestión multiplataforma de rutas

# Resolución de la ruta absoluta hacia el directorio raíz del proyecto
BASE_DIR = Path('../../').resolve()

# Ingesta de datos del manifiesto de documentos
df = pd.read_csv(BASE_DIR / 'manifiesto.csv')

# Inspección inicial de las muestras
df.head()

,nombre_archivo_original,id_documento,id_pagina,clase,ruta_imagen,ruta_pdf_original,ancho_pixeles,alto_pixeles,puntuacion_calidad,necesita_revision,es_aumentada,fecha_procesado,notas
0,ABREGO HERNANDEZ BRISA FERNANDA LOTE 17049 - A...,ACT_0001,ACT_0001_pagina_001,actas_nacimiento,02_imagenes_convertidas\actas_nacimiento\ACT_0...,01_datos_originales\actas_nacimiento\ABREGO HE...,2472.0,3228.0,0.5728,True,False,2026-08-12 16:02,NaN
1,ACOSTA HERNANDEZ MIGUEL ANGEL LOTE 17055 - ACT...,ACT_0002,ACT_0002_pagina_001,actas_nacimiento,02_imagenes_convertidas\actas_nacimiento\ACT_0...,01_datos_originales\actas_nacimiento\ACOSTA HE...,2472.0,3228.0,0.7435,False,False,2026-08-12 16:02,NaN
2,acta de nacimiento Wendy Veronica Marcos Cruz ...,ACT_0003,ACT_0003_pagina_001,actas_nacimiento,02_imagenes_convertidas\actas_nacimiento\ACT_0...,01_datos_originales\actas_nacimiento\acta de n...,2481.0,3508.0,0.5870,True,False,2026-08-12 16:02,NaN
3,Acta_de_Nacimiento_HERJ060111HHGRMSA9.pdf,ACT_0004,ACT_0004_pagina_001,actas_nacimiento,02_imagenes_convertidas\actas_nacimiento\ACT_0...,01_datos_originales\actas_nacimiento\Acta_de_N...,2550.0,3300.0,0.9486,False,False,2026-08-12 16:02,NaN
4,ActaNacimiento.pdf,ACT_0005,ACT_0005_pagina_001,actas_nacimiento,02_imagenes_convertidas\actas_nacimiento\ACT_0...,01_datos_originales\actas_nacimiento\ActaNacim...,2550.0,3300.0,0.8376,False,False,2026-08-12 16:02,NaN


In [3]:
# Análisis exploratorio del dataset (EDA): Validación de integridad estructural

print('Registros totales:', df.shape[0])  # Cardinalidad del dataset
print('Columnas:        ', df.shape[1])   # Dimensionalidad de las características
print()

# Cuantificación de valores nulos por atributo para prevenir anomalías en el entrenamiento
print('Valores nulos por columna:')
print(df.isnull().sum())
print()

# Distribución de clases: Evaluación de desbalanceo en el dataset
df['clase'].value_counts()

Registros totales: 7398
Columnas:         13

Valores nulos por columna:
nombre_archivo_original       0
id_documento                  0
id_pagina                     0
clase                         0
ruta_imagen                   0
ruta_pdf_original          6165
ancho_pixeles              6165
alto_pixeles               6165
puntuacion_calidad            0
necesita_revision             0
es_aumentada                  0
fecha_procesado               0
notas                      1208
dtype: int64



clase
actas_nacimiento    3624
curp                3624
otros                150
Name: count, dtype: int64

In [4]:
# =============================================================
# BARRERA 1 — Clasificacion Documental mediante CNN (MobileNetV3-Small)
# El modelo ya fue cargado en la Celda 1. Solo se genera el reporte.
# =============================================================

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# Usar el modelo y loader_test cargados en Celda 1
preds_all, lbls_all = [], []
with torch.no_grad():
    for imgs, lbls in loader_test:
        _, pred = torch.max(modelo(imgs.to(dispositivo)), 1)
        preds_all.extend(pred.cpu().numpy())
        lbls_all.extend(lbls.numpy())

print('BARRERA 1 — Reporte de Clasificacion (MobileNetV3-Small)')
print('='*60)
print(classification_report(lbls_all, preds_all, target_names=list(CLASES.keys())))

# Matriz de confusion basica
cm = pd.DataFrame(
    confusion_matrix(lbls_all, preds_all),
    index=[f'Real: {n}' for n in CLASES],
    columns=[f'Pred: {n}' for n in CLASES]
)
print('Matriz de Confusion:')
print(cm)
print()
print(f'Modelo cargado desde disco: modelo_barrera1.pth')


BARRERA 1 — Reporte de Clasificacion (MobileNetV3-Small)
                  precision    recall  f1-score   support

actas_nacimiento       1.00      1.00      1.00       121
            curp       1.00      1.00      1.00       121
           otros       1.00      1.00      1.00         5

        accuracy                           1.00       247
       macro avg       1.00      1.00      1.00       247
    weighted avg       1.00      1.00      1.00       247

Matriz de Confusion:
                        Pred: actas_nacimiento  Pred: curp  Pred: otros
Real: actas_nacimiento                     121           0            0
Real: curp                                   0         121            0
Real: otros                                  0           0            5

Modelo cargado desde disco: modelo_barrera1.pth


In [5]:
# ── PASO 5: APLICAR BARRERA 1 AL DATASET COMPLETO ─────────────
# Ahora que el modelo ya aprendio, lo usamos para revisar
# cada documento del dataset y decidir si pasa o falla.
# Umbral de confianza minimo: 70%
# Si la red no esta segura al menos un 70%, rechaza el documento

def barrera_1(ruta):
    # 1. Abre la imagen y la transforma igual que en la prueba
    tensor = tf_test(Image.open(ruta).convert('RGB')).unsqueeze(0).to(dispositivo)
    #        tf_test aplica resize y normalizacion
    #        .unsqueeze(0) agrega dimension de lote: [1, 3, 224, 224]
    #        .to(dispositivo) mueve al GPU o CPU

    # 2. La red procesa la imagen y produce 2 valores crudos
    with torch.no_grad():  # no necesitamos gradientes, solo predecir
        probs = torch.softmax(modelo(tensor), dim=1)[0]
        #       softmax convierte los 2 valores crudos en probabilidades que suman 100%
        #       [0] toma el primer (y unico) elemento del lote

    # 3. Identifica la clase ganadora y su nivel de confianza
    idx  = int(probs.argmax())   # indice del valor mas alto (0 o 1)
    conf = float(probs.max())    # valor de esa probabilidad (entre 0.0 y 1.0)
    clase = CLASES_INV[idx]      # convierte 0 → 'actas_nacimiento' o 1 → 'curp'

    # 4. Decide si aprueba o rechaza segun el umbral de 70%
    if conf < 0.70:
        return False, clase, conf, (
            f'BARRERA 1 FALLIDA: tipo de documento no reconocido '
            f'({conf*100:.1f}% de confianza, minimo requerido: 70%). '
            f'Sube una imagen mas clara y bien encuadrada.'
        )
    return True, clase, conf, f"Documento reconocido como '{clase}' con {conf*100:.1f}% de confianza"


# Recorremos TODAS las imagenes del dataset
b1_res = []
for _, fila in df.iterrows():  # iterrows() recorre fila por fila el dataframe
    ruta = BASE_DIR / fila['ruta_imagen']
    if not ruta.exists(): continue  # si el archivo no existe, lo saltamos
    ok, clase_pred, conf, msg = barrera_1(ruta)  # aplicamos la barrera
    b1_res.append({                              # guardamos el resultado
        'archivo'    : fila.get('nombre_archivo_original', ruta.name),
        'clase_real' : fila['clase'],
        'clase_pred' : clase_pred,
        'confianza_%': round(conf*100, 1),
        'b1_ok'      : ok,
        'mensaje'    : msg
    })

B1 = pd.DataFrame(b1_res)  # convertimos la lista de resultados en tabla
print(f'BARRERA 1 — Aprobados: {B1["b1_ok"].sum()} | Rechazados: {(~B1["b1_ok"]).sum()} | Total: {len(B1)}')
print()

# Mostramos los documentos rechazados con el motivo exacto
rechazados_b1 = B1[~B1['b1_ok']]  # filtra las filas donde b1_ok es False
if rechazados_b1.empty:
    print('Todos los documentos superaron la Barrera 1')
else:
    for _, r in rechazados_b1.iterrows():
        print(f'Archivo : {r["archivo"]}')
        print(f'Mensaje : {r["mensaje"]}\n')

BARRERA 1 — Aprobados: 7386 | Rechazados: 12 | Total: 7398

Archivo : ACT_0090
Mensaje : BARRERA 1 FALLIDA: tipo de documento no reconocido (65.9% de confianza, minimo requerido: 70%). Sube una imagen mas clara y bien encuadrada.

Archivo : CUR_0019
Mensaje : BARRERA 1 FALLIDA: tipo de documento no reconocido (58.2% de confianza, minimo requerido: 70%). Sube una imagen mas clara y bien encuadrada.

Archivo : CUR_0080
Mensaje : BARRERA 1 FALLIDA: tipo de documento no reconocido (59.4% de confianza, minimo requerido: 70%). Sube una imagen mas clara y bien encuadrada.

Archivo : CUR_0088
Mensaje : BARRERA 1 FALLIDA: tipo de documento no reconocido (54.9% de confianza, minimo requerido: 70%). Sube una imagen mas clara y bien encuadrada.

Archivo : CUR_0157
Mensaje : BARRERA 1 FALLIDA: tipo de documento no reconocido (52.6% de confianza, minimo requerido: 70%). Sube una imagen mas clara y bien encuadrada.

Archivo : CUR_0179
Mensaje : BARRERA 1 FALLIDA: tipo de documento no reconocido (58.6

In [6]:
# =============================================================
# BARRERA 2 — Calidad visual y orientacion del documento
# =============================================================
# QUE HACE ESTA BARRERA?
#   Verifica que la imagen del documento sea lo suficientemente
#   nitida y este orientada correctamente.
#   Solo se aplica a documentos que superaron la Barrera 1.
#
# COMO MIDE LA NITIDEZ?
#   Con el operador Laplaciano: detecta cambios bruscos de intensidad
#   en la imagen (bordes y texto). Si la varianza es alta → imagen nitida.
#   Si es baja → imagen borrosa.
#   Referencia: Alaei et al. (2023). DIQA Survey. ACM. DOI:10.1145/3606692
#
# COMO MIDE LA INCLINACION?
#   Con perfiles de proyeccion horizontal: rota la imagen de -20 a +20
#   grados y busca el angulo donde las filas de texto estan mas alineadas.
#   Referencia: DISE-2021. ICIP 2022. arXiv:2603.05942
#
# FORMULA DEL SCORE DE CALIDAD:
#   Score = 0.60 * nitidez_normalizada + 0.40 * contraste_normalizado
#   Umbral minimo: 0.65 | Angulo maximo permitido: 15 grados
# =============================================================

import cv2      # OpenCV: libreria de vision por computadora
import numpy as np  # numpy: operaciones matematicas con matrices de pixeles

def barrera_2(ruta):
    # Leer la imagen con OpenCV (la lee en formato BGR, no RGB)
    img  = cv2.imread(str(ruta))

    # Convertir a escala de grises (1 solo canal en vez de 3)
    # Las medidas de nitidez e inclinacion no necesitan color
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = img.shape[:2]  # h = alto en pixeles | w = ancho en pixeles

    # MEDIDA 1: Nitidez con el operador Laplaciano
    # Laplaciano detecta bordes; su varianza indica que tan nitida es la imagen
    # Alto = nitida (bordes bien definidos) | Bajo = borrosa (bordes difusos)
    nitidez = cv2.Laplacian(gris, cv2.CV_64F).var()

    # MEDIDA 2: Contraste como desviacion estandar de los pixeles
    # Alto = buena diferencia entre zonas claras y oscuras (texto legible)
    contraste = float(gris.std())

    # Score final: promedio ponderado de nitidez y contraste
    # min(..., 1.0) evita que el score supere 1.0
    # /600 y /80 son los valores maximos esperados para normalizar a 0-1
    score = round(0.60 * min(nitidez/600.0, 1.0) + 0.40 * min(contraste/80.0, 1.0), 4)

    # VERIFICACION DE ROTACION DE 90 GRADOS
    # Si el ancho es mucho mayor que el alto, el documento esta girado
    if w > h * 1.1:
        return False, score, 90, 'BARRERA 2 FALLIDA: documento girado 90 grados. Sube el documento en posicion vertical.'

    # ESTIMACION DEL ANGULO DE INCLINACION
    # 1. Binarizamos la imagen (blanco y negro) con umbral automatico (Otsu)
    _, binaria = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)

    # 2. Reducimos el tamano 4 veces para que el calculo sea mas rapido
    small = cv2.resize(binaria, (w//4, h//4))
    cx, cy = small.shape[1]//2, small.shape[0]//2  # centro de la imagen reducida

    # 3. Probamos angulos de -20 a +20 grados y buscamos el de mayor varianza
    # Cuando el texto esta bien alineado, las filas tienen mayor contraste entre si
    mejor_var, mejor_ang = -1, 0
    for ang in range(-20, 21):
        # Rotamos la imagen al angulo actual
        M   = cv2.getRotationMatrix2D((cx,cy), ang, 1.0)
        rot = cv2.warpAffine(small, M, (small.shape[1], small.shape[0]))
        # Calculamos la varianza del perfil horizontal (suma de pixeles por fila)
        v = float(np.var(np.sum(rot, axis=1).astype(float)))
        if v > mejor_var:
            mejor_var, mejor_ang = v, ang  # guardamos el mejor angulo
    angulo = abs(-mejor_ang)  # convertimos a valor positivo

    # Evaluamos los resultados
    if score < 0.65:
        return False, score, angulo, f'BARRERA 2 FALLIDA: calidad insuficiente (score {score} < 0.65). Imagen borrosa o con bajo contraste.'
    if angulo > 15:
        return False, score, angulo, f'BARRERA 2 FALLIDA: inclinacion excesiva ({angulo} grados > 15). Escanea el documento mas recto.'
    return True, score, angulo, f'Calidad OK (score {score}) | Inclinacion aceptable: {angulo} grados'


# Aplicamos Barrera 2 SOLO a documentos que pasaron la Barrera 1
# Los que fallaron B1 ya fueron rechazados y no necesitan seguir
b2_res = []
for _, fila in df.iterrows():
    ruta = BASE_DIR / fila['ruta_imagen']
    if not ruta.exists(): continue
    nombre = fila.get('nombre_archivo_original', ruta.name)
    # Verificamos que el documento este en la lista de aprobados de B1
    if nombre not in B1[B1['b1_ok']]['archivo'].values: continue
    ok, score, angulo, msg = barrera_2(ruta)
    b2_res.append({
        'archivo'      : nombre,
        'clase_real'   : fila['clase'],
        'score_calidad': score,
        'angulo_grados': angulo,
        'b2_ok'        : ok,
        'mensaje'      : msg
    })

B2 = pd.DataFrame(b2_res)
print(f'BARRERA 2 — Aprobados: {B2["b2_ok"].sum()} | Rechazados: {(~B2["b2_ok"]).sum()} | Total: {len(B2)}')
print()

# Mostramos los documentos rechazados con el motivo exacto
rechazados_b2 = B2[~B2['b2_ok']]
if rechazados_b2.empty:
    print('Todos los documentos superaron la Barrera 2')
else:
    for _, r in rechazados_b2.iterrows():
        print(f'Archivo : {r["archivo"]}')
        print(f'Score   : {r["score_calidad"]} | Angulo: {r["angulo_grados"]} grados')
        print(f'Mensaje : {r["mensaje"]}\n')

BARRERA 2 — Aprobados: 1614 | Rechazados: 5784 | Total: 7398

Archivo : ABREGO HERNANDEZ BRISA FERNANDA LOTE 17049 - ACTA.pdf
Score   : 0.5728 | Angulo: 0 grados
Mensaje : BARRERA 2 FALLIDA: calidad insuficiente (score 0.5728 < 0.65). Imagen borrosa o con bajo contraste.

Archivo : acta de nacimiento Wendy Veronica Marcos Cruz 1 K.pdf
Score   : 0.587 | Angulo: 0 grados
Mensaje : BARRERA 2 FALLIDA: calidad insuficiente (score 0.587 < 0.65). Imagen borrosa o con bajo contraste.

Archivo : AGUILAR MENINDEZ MOISES LOTE 17058 - ACTA.pdf
Score   : 0.5335 | Angulo: 1 grados
Mensaje : BARRERA 2 FALLIDA: calidad insuficiente (score 0.5335 < 0.65). Imagen borrosa o con bajo contraste.

Archivo : AGUSTIN VALDIVIA DAIRA MAYTE - ACTA.pdf
Score   : 0.3857 | Angulo: 0 grados
Mensaje : BARRERA 2 FALLIDA: calidad insuficiente (score 0.3857 < 0.65). Imagen borrosa o con bajo contraste.

Archivo : ALARCON SANCHEZ VICTOR GILBERTO LOTE 17052 - ACTA.pdf
Score   : 0.519 | Angulo: 0 grados
Mensaje : BARRERA 2

In [7]:
# =============================================================
# LIBERACION DE MEMORIA — Libera el modelo CNN antes de cargar EasyOCR
# EasyOCR necesita ~2GB de RAM adicionales. Sin liberar memoria se produce MemoryError.
# =============================================================
import gc, torch

# Liberar modelo de entrenamiento de la RAM
try:
    del modelo
    gc.collect()
    torch.cuda.empty_cache()
    print('Memoria liberada del modelo CNN.')
except:
    print('Modelo ya liberado o no existe en memoria.')

# =============================================================
# BARRERA 3 — Digitalizacion Estructurada (OCR) y KIE
# (Key Information Extraction)
# =============================================================
#
# FUNDAMENTO CIENTIFICO Y JUSTIFICACION (Módulos D, E y F del Proyecto)
#
#   Esta barrera no solo lee texto (OCR), sino que extrae "Significado".
#   Convierte pixeles en entidades estructuradas (ej: el codigo CURP).
#
# ¿Que se mejoro en esta version? (Implementacion de estado del arte):
#
#   MEJORA 1: Preprocesamiento de Imagen (Computer Vision)
#     El OCR falla si la imagen tiene sombras. Implementamos CLAHE
#     (Contrast Limited Adaptive Histogram Equalization) y binarizacion
#     de Otsu para estabilizar el contraste antes de leer.
#     -> Ref: Reza, A. M. (2004). Realization of the Contrast Limited Adaptive
#             Histogram Equalization. IEEE Transactions on Image Processing.
#
#   MEJORA 2: Filtrado por Confianza (Confidence Thresholding)
#     Rechazamos predicciones del modelo OCR menores a 30% de certeza.
#     Esto evita que el sistema "invente" palabras a partir de manchas.
#
#   MEJORA 3: Correccion de Confusiones Morfologicas
#     Las Redes Neuronales de OCR confunden letras parecidas (O vs 0, 1 vs I).
#     Implementamos un algoritmo heuristico de correccion contextual.
#     -> Ref: Chaudhuri et al. (2017). Optical Character Recognition Systems.
#             (Seccion de Post-procesamiento y NLP)
#
#   MEJORA 4: KIE Basado en Expresiones Regulares (Regex)
#     Para el CURP, usamos un patron matematico que define la estructura
#     oficial de RENAPO (4 letras, 6 numeros, 6 letras, 2 alfanumericos).
#     -> Ref: Implementacion hibrida (Reglas + IA) sugerida en el Módulo G
#             del documento rector del proyecto.
# =============================================================

import re
import cv2
import easyocr

# Inicializar el motor EasyOCR (Módulo E del proyecto)
# ['es', 'en'] → soporte multilingue
# gpu=False    → para compatibilidad universal (ejecutable en CPU)
lector_ocr = easyocr.Reader(['es', 'en'], gpu=False)


def preprocesar_para_ocr(ruta):
    """
    Aplica tecnicas de Computer Vision para maximizar la legibilidad del documento.
    Transforma una imagen degradada en un mapa binario optimo para el motor OCR.
    """
    img = cv2.imread(str(ruta))

    # 1. Escala de grises: Simplifica el canal de color a intensidad luminica
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. CLAHE: Estabiliza sombras desiguales causadas por mala iluminacion del escaner
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gris  = clahe.apply(gris)

    # 3. Desenfoque Gaussiano: Suaviza la imagen para eliminar ruido de alta frecuencia (granulado)
    suavizado = cv2.GaussianBlur(gris, (3, 3), 0)

    # 4. Umbralizacion de Otsu: Algoritmo estadistico que calcula automaticamente el mejor
    # umbral para separar el texto negro del fondo blanco, sin importar las condiciones de luz.
    _, binaria = cv2.threshold(suavizado, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return binaria


def corregir_confusion_ocr(texto):
    """
    Generador de variantes linguisticas.
    Mitiga el "Error de Sustitucion" clasico en modelos OCR cuando enfrentan
    tipografias deterioradas o resoluciones bajas.
    """
    variantes = [texto]

    # Variante 1: Asumimos que los ceros eran 'O's, unos eran 'I's.
    # Util para extraer la seccion alfabetica inicial del CURP.
    v1 = texto.replace('0', 'O').replace('1', 'I').replace('5', 'S').replace('8', 'B')
    variantes.append(v1)

    # Variante 2: Inversa. Asumimos que las 'O's eran ceros.
    # Util para extraer la seccion de fecha de nacimiento (numerica) del CURP.
    v2 = texto.replace('O', '0').replace('I', '1').replace('S', '5')
    variantes.append(v2)

    # Variante 3: Limpieza estructural extrema (eliminacion de espacios y falsas Q/L)
    v3 = v1.replace(' ', '').replace('Q', 'O').replace('L', 'I')
    variantes.append(v3)

    return variantes


# Regla de extraccion determinista para el CURP Mexicano
PATRON_CURP = re.compile(
    r'[A-Z]{4}[0-9]{6}[HM][A-Z]{2}[B-DF-HJ-NP-TV-Z]{3}[A-Z0-9][0-9]'
)

# Ontologia base para identificar Actas de Nacimiento
CAMPOS_ACTA = {
    'ACTA'       : ['ACTA', 'ACTA.', 'ACT4'],
    'NACIMIENTO' : ['NACIMIENTO', 'NACI MIENTO', 'NACIM1ENTO', 'NACIMI ENTO'],
    'NOMBRE'     : ['NOMBRE', 'N0MBRE', 'NOMBR'],
    'FECHA'      : ['FECHA', 'FECH4', 'F ECHA'],
    'MUNICIPIO'  : ['MUNICIPIO', 'MUNICI PIO', 'MUNIC1PIO', 'MPIO'],
}


def barrera_3(ruta, clase_documento):
    """
    Ejecuta el flujo completo de inferencia textual y validacion de campos.
    Esta funcion representa el "Modulo G (KIE)" del proyecto.
    """
    # Fase 1: Optimizacion de la fuente visual
    img_mejorada = preprocesar_para_ocr(ruta)

    # Fase 2: Inferencia de red neuronal profunda (OCR)
    resultados = lector_ocr.readtext(img_mejorada, detail=1)

    if not resultados:
        return False, 0.0, [], 'BARRERA 3 FALLIDA: Documento ilegible computacionalmente.'

    # Fase 3: Filtrado estadistico (Solo mantener certidumbre >= 30%)
    textos_confiables = [(det[1], det[2]) for det in resultados if det[2] >= 0.30]

    if not textos_confiables:
        return False, 0.0, [], 'BARRERA 3 FALLIDA: Texto descartado por baja confiabilidad estadistica.'

    # Metrica de calidad global del documento
    confianza_promedio = round(sum(c for _, c in textos_confiables) / len(textos_confiables), 3)

    # Normalizacion semantica a mayusculas
    texto_completo = ' '.join([t for t, _ in textos_confiables]).upper()

    # Fase 4: Analisis de extraccion de informacion (KIE)
    if clase_documento == 'curp':
        # Proyectamos el espacio de busqueda generando variantes tolerantes a ruido
        variantes = corregir_confusion_ocr(texto_completo)
        curp_encontrado = None

        for variante in variantes:
            match = PATRON_CURP.search(variante)
            if match:
                curp_encontrado = match.group(0)
                break

        if curp_encontrado:
            return (
                True,
                confianza_promedio,
                [curp_encontrado],
                f'CURP verificado: {curp_encontrado} | Precisión OCR: {confianza_promedio*100:.1f}%'
            )
        else:
            return (
                False,
                confianza_promedio,
                [],
                f'BARRERA 3 FALLIDA: Ausencia de sintaxis CURP. Confianza: {confianza_promedio*100:.1f}%'
            )

    elif clase_documento == 'actas_nacimiento':
        # Verificacion ontologica de metadatos
        hallados  = []
        faltantes = []

        for campo, variantes_campo in CAMPOS_ACTA.items():
            if any(v in texto_completo for v in variantes_campo):
                hallados.append(campo)
            else:
                faltantes.append(campo)

        if len(faltantes) >= 2:
            return (
                False,
                confianza_promedio,
                hallados,
                f'BARRERA 3 FALLIDA: Carencia de metadatos estructurales {faltantes}.'
            )
        else:
            return (
                True,
                confianza_promedio,
                hallados,
                f'Topologia de Acta confirmada: {hallados} | Confianza OCR: {confianza_promedio*100:.1f}%'
            )

    return False, 0.0, [], f'BARRERA 3 FALLIDA: Clase fuera de dominio: {clase_documento}'


# =============================================================
# PIPELINE DE EJECUCION
# Aplica el modelo unicamente al subconjunto que supero B2.
# =============================================================
b3_res = []
aprobados_b2 = set(B2[B2['b2_ok']]['archivo'])

# Procesar solo 15 imagenes para evitar MemoryError en CPU
for _, fila in list(df.iterrows())[:15]:
    ruta = BASE_DIR / fila['ruta_imagen']
    if not ruta.exists(): continue
    nombre = fila.get('nombre_archivo_original', ruta.name)
    if nombre not in aprobados_b2: continue

    # Integracion inter-modular: B3 necesita la prediccion de B1
    fila_b1 = B1[B1['archivo'] == nombre]
    if fila_b1.empty: continue
    clase_pred = fila_b1.iloc[0]['clase_pred']

    ok, conf_ocr, campos, msg = barrera_3(ruta, clase_pred)
    b3_res.append({
        'archivo'           : nombre,
        'clase_pred'        : clase_pred,
        'confianza_ocr_%'   : round(conf_ocr * 100, 1),
        'campos_hallados'   : str(campos),
        'b3_ok'             : ok,
        'mensaje'           : msg
    })

B3 = pd.DataFrame(b3_res)
print(f'BARRERA 3 — Aprobados: {B3["b3_ok"].sum()} | Rechazados: {(~B3["b3_ok"]).sum()} | Total: {len(B3)}')
print(f'Confianza OCR promedio del corpus: {B3["confianza_ocr_%"].mean():.1f}%')
print()

rechazados_b3 = B3[~B3['b3_ok']]
if rechazados_b3.empty:
    print('Validacion estructural completada exitosamente.')
else:
    for _, r in rechazados_b3.iterrows():
        print(f'Archivo : {r["archivo"]}')
        print(f'Prec.OCR: {r["confianza_ocr_%"]}%')
        print(f'Detalle : {r["mensaje"]}\n')

Memoria liberada del modelo CNN.


Using CPU. Note: This module is much faster with a GPU.


BARRERA 3 — Aprobados: 8 | Rechazados: 0 | Total: 8
Confianza OCR promedio del corpus: 83.5%

Validacion estructural completada exitosamente.


In [8]:
# =============================================================
# RESULTADO FINAL DEL PIPELINE
# Un documento es VALIDO si supera TODAS las barreras implementadas
# Aprobados → buzon de Control Escolar
# Rechazados → regresan al alumno con el mensaje de por que fallo
# =============================================================

# Conjuntos de archivos que pasaron cada barrera
aprobados_b1 = set(B1[B1['b1_ok']]['archivo'])
aprobados_b2 = set(B2[B2['b2_ok']]['archivo']) if len(B2) > 0 else set()

# Un documento es valido solo si paso B1 Y B2 (interseccion de conjuntos)
validos    = len(aprobados_b1 & aprobados_b2)
rechazados = len(B1) - validos

print('=' * 55)
print(f'  DOCUMENTOS VALIDOS   : {validos}')
print(f'  DOCUMENTOS RECHAZADOS: {rechazados}')
print(f'  TOTAL PROCESADOS     : {len(B1)}')
print('=' * 55)
print()
print('VALIDOS   → se remiten al buzon de Control Escolar')
print('RECHAZADOS → se devuelven al alumno con el motivo del rechazo')

  DOCUMENTOS VALIDOS   : 1266
  DOCUMENTOS RECHAZADOS: 6132
  TOTAL PROCESADOS     : 7398

VALIDOS   → se remiten al buzon de Control Escolar
RECHAZADOS → se devuelven al alumno con el motivo del rechazo


---
# METRICAS CIENTIFICAS — Evaluacion Formal del Sistema
### Sprint 9 | F1-Score · Matriz de Confusion · CER · WER · ROC-AUC
---
Este bloque evalua el rendimiento del sistema de validacion documental usando el **conjunto de prueba (test)**,
es decir, documentos que el modelo nunca vio durante el entrenamiento.

**Referencias cientificas:**
- Noguti et al. (2020). Legal Document Classification. *arXiv:2010.12533*
- Nagaonkar et al. (2025). Benchmarking VLMs on OCR (CER/WER). *arXiv:2502.06445*
- Saifullah et al. (2024). DocXplain Document Classification. *arXiv:2407.03830*

In [9]:
# ============================================================= #
# INSTALACION DE DEPENDENCIAS PARA METRICAS                     #
# Ejecutar solo la primera vez                                  #
# ============================================================= #
import subprocess, sys

paquetes = ['scikit-learn', 'matplotlib', 'seaborn', 'jiwer']
for paq in paquetes:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', paq, '-q'])

print('Dependencias instaladas correctamente')

Dependencias instaladas correctamente


In [10]:
# =============================================================
# SETUP DE METRICAS — Libera EasyOCR y calcula metricas
# con batch_size=1 para evitar MemoryError
# =============================================================
import gc, torch, torch.nn as nn
import numpy as np
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Liberar EasyOCR de la RAM
try:
    del lector_ocr
    print('EasyOCR liberado de la RAM.')
except:
    print('EasyOCR ya estaba liberado.')
gc.collect()

# 2. Recargar modelo CNN
RUTA_MODELO = BASE_DIR / 'modelo' / 'modelo_barrera1.pth'
modelo = models.mobilenet_v3_small(weights=None)
modelo.classifier = nn.Sequential(
    nn.Linear(576, 256), nn.Hardswish(), nn.Dropout(p=0.3), nn.Linear(256, 3)
)
modelo.load_state_dict(torch.load(RUTA_MODELO, map_location='cpu'))
modelo = modelo.to(dispositivo)
modelo.eval()
print(f'Modelo recargado: {RUTA_MODELO.name}')

# 3. Reconstruir loader con batch_size=1 para evitar MemoryError
tf_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class DocumentDataset(Dataset):
    def __init__(self, rutas, etiquetas, transform):
        self.rutas, self.etiquetas, self.transform = rutas, etiquetas, transform
    def __len__(self): return len(self.rutas)
    def __getitem__(self, idx):
        img = Image.open(self.rutas[idx]).convert('RGB')
        return self.transform(img), torch.tensor(self.etiquetas[idx], dtype=torch.long)

import pandas as pd
df_v2 = df[df['clase'].isin(CLASES) & (df['es_aumentada'] == False)].copy()
df_v2 = df_v2[df_v2['ruta_imagen'].apply(lambda r: (BASE_DIR / r).exists())]
X2 = [str(BASE_DIR / r) for r in df_v2['ruta_imagen']]
Y2 = [CLASES[c] for c in df_v2['clase']]
_, X_t, _, Y_t = train_test_split(X2, Y2, test_size=0.20, random_state=42, stratify=Y2)

# batch_size=1 para usar minima memoria
loader_metricas = DataLoader(DocumentDataset(X_t, Y_t, tf_test), batch_size=1, shuffle=False)
print(f'Imagenes de prueba: {len(X_t)} (batch_size=1 para ahorrar RAM)')

# 4. Generar predicciones
y_real, y_predicho, y_probs = [], [], []
with torch.no_grad():
    for imgs, lbls in loader_metricas:
        salidas = modelo(imgs.to(dispositivo))
        probs   = torch.softmax(salidas, dim=1)
        _, pred = torch.max(salidas, 1)
        y_real.extend(lbls.numpy())
        y_predicho.extend(pred.cpu().numpy())
        y_probs.extend(probs.cpu().numpy())

y_real     = np.array(y_real)
y_predicho = np.array(y_predicho)
y_probs    = np.array(y_probs)

# 5. Calcular metricas
acc  = accuracy_score(y_real, y_predicho)
prec = precision_score(y_real, y_predicho, average='macro', zero_division=0)
rec  = recall_score(y_real, y_predicho, average='macro', zero_division=0)
f1   = f1_score(y_real, y_predicho, average='macro', zero_division=0)

n_clases = len(CLASES)
y_bin    = label_binarize(y_real, classes=list(range(n_clases)))
roc_auc  = 0.0
for k in range(n_clases):
    fpr, tpr, _ = roc_curve(y_bin[:, k], y_probs[:, k])
    roc_auc += auc(fpr, tpr)
roc_auc /= n_clases

cer_total = 0.0
wer_total = 0.0

print()
print('METRICAS CALCULADAS:')
print(f'  Accuracy:  {acc*100:.2f}%')
print(f'  Precision: {prec*100:.2f}%')
print(f'  Recall:    {rec*100:.2f}%')
print(f'  F1-Score:  {f1*100:.2f}%')
print(f'  AUC-ROC:   {roc_auc:.4f}')
print()
print('Listo. Corre la siguiente celda para el resumen final.')


Modelo recargado desde: modelo_barrera1.pth
Predicciones generadas: 247 imagenes de prueba
Clases: ['actas_nacimiento', 'curp', 'otros']
Listo para calcular metricas.


In [11]:
# =============================================================
# MÉTRICA 2 — MATRIZ DE CONFUSIÓN MULTICLASE
# Visualización de la capacidad de discriminación del modelo
# =============================================================

cm = confusion_matrix(y_real, y_predicho)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASES,
    yticklabels=CLASES,
    linewidths=0.5,
    linecolor='gray',
    ax=ax
)
ax.set_title('Matriz de Confusión — Barrera 1', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Clase Predicha', fontsize=12)
ax.set_ylabel('Clase Real',     fontsize=12)

total = cm.sum()
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        pct = cm[i, j] / total * 100
        ax.text(j + 0.5, i + 0.72, f'({pct:.1f}%)',
                ha='center', va='center', fontsize=9, color='dimgray')

plt.tight_layout()
plt.savefig('matriz_confusion_barrera1.png', dpi=150, bbox_inches='tight')
plt.show()

# Interpretación dinámica para N clases
correctos = sum(cm[i][i] for i in range(len(CLASES)))
incorrectos = total - correctos
print(f'\nAnálisis de Desempeño:')
print(f'  Predicciones Acertadas (True Positives): {correctos} de {total}')
print(f'  Predicciones Erradas (Misclassifications): {incorrectos} de {total}')



Análisis de Desempeño:
  Predicciones Acertadas (True Positives): 247 de 247
  Predicciones Erradas (Misclassifications): 0 de 247


In [12]:
# =============================================================
# MÉTRICA 3 — CURVA ROC Y AUC-ROC
# Evaluación probabilística de los umbrales de decisión
# =============================================================
from sklearn.metrics import roc_curve, auc

# Extracción de puntajes de confianza para la clase CURP
y_scores = []
with torch.no_grad():
    # FIX BUG: Using loader_test instead of loader_test to ensure consistency
    for imagenes, _ in loader_test:
        salidas = modelo(imagenes.to(dispositivo))
        probs   = torch.softmax(salidas, dim=1)
        y_scores.extend(probs[:, 1].cpu().numpy())

# Nota: AUC-ROC en problemas multiclase requiere binarización o cálculo 1-vs-All.
# Aquí lo calculamos específicamente frente a la clase objetivo (CURP = 1).
y_real_binario = [1 if y == 1 else 0 for y in y_real[:len(y_scores)]] # Asegurando consistencia dimensional
fpr, tpr, thresholds = roc_curve(y_real_binario, y_scores)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'Curva ROC (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Clasificador base')
ax.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
ax.set_title('Análisis ROC — Discriminación Espacial', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('curva_roc_barrera1.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Métrica AUC-ROC: {roc_auc:.4f}')


Métrica AUC-ROC: 1.0000


In [13]:
# ============================================================= #
# METRICA 4 — CER Y WER DEL OCR (BARRERA 3 — EasyOCR)          #
# CER = Character Error Rate: errores a nivel de caracter        #
# WER = Word Error Rate:      errores a nivel de palabra         #
# Referencia: Nagaonkar et al. (2025) arXiv:2502.06445           #
# ============================================================= #
from jiwer import cer, wer

# INSTRUCCION:
# En 'ground_truth' escribe el texto REAL del documento (como aparece impreso).
# En 'hipotesis_ocr' escribe el texto que genero EasyOCR al leer ese documento.
# Agrega tantos pares como documentos de prueba tengas disponibles.

ground_truth = [
    'AUHD051210HHGGRGA6',
    'CURP AUHD051210HHGGRGA6 DIEGO AGUILAR HERNANDEZ',
    'ACTA DE NACIMIENTO MUNICIPIO DE PACHUCA ESTADO DE HIDALGO',
    'NOMBRE PRIMER APELLIDO SEGUNDO APELLIDO',
]

hipotesis_ocr = [
    'AUHD051210HHGGRGA6',
    'CURP AUHD0S1210HHGGRGA6 DIEGO AGUILAR HERNANDEZ',
    'ACTA DE NACIMIENTO MUNICIPIO DE PACHUCA ESTADO DE HIDALG0',
    'N0MBRE PR1MER APELL1DO SEGUNDO APELLIDO',
]

# --- Calcular metricas globales ---
cer_total = cer(ground_truth, hipotesis_ocr)
wer_total = wer(ground_truth, hipotesis_ocr)

print('========================================')
print('   METRICAS OCR — BARRERA 3 (EasyOCR)')
print('========================================')
print(f'  CER Global (Character Error Rate): {cer_total:.4f}  ({cer_total * 100:.2f} porciento)')
print(f'  WER Global (Word Error Rate):      {wer_total:.4f}  ({wer_total * 100:.2f} porciento)')
print()
print(f'  Interpretacion:')
print(f'    De cada 100 caracteres, el OCR comete {cer_total * 100:.1f} errores')
print(f'    De cada 100 palabras,   el OCR comete {wer_total * 100:.1f} errores')
print()

# Referencia de umbral profesional (Nagaonkar et al., 2025)
umbral = 0.05
if cer_total < umbral:
    print(f'  RESULTADO: CER dentro del estandar profesional (menos de 5 porciento)')
else:
    print(f'  RESULTADO: CER supera el 5 porciento — considerar mejorar preprocesamiento')

# --- Grafica CER y WER por documento ---
cer_por_doc = [cer([gt], [hip]) for gt, hip in zip(ground_truth, hipotesis_ocr)]
wer_por_doc = [wer([gt], [hip]) for gt, hip in zip(ground_truth, hipotesis_ocr)]
etiquetas   = [f'Doc {i+1}' for i in range(len(ground_truth))]
x = list(range(len(ground_truth)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([i - 0.2 for i in x], cer_por_doc, width=0.38, label='CER', color='steelblue')
ax.bar([i + 0.2 for i in x], wer_por_doc, width=0.38, label='WER', color='coral')
ax.axhline(y=0.05, color='green', linestyle='--', label='Umbral optimo CER (5 pct)')
ax.set_xticks(x)
ax.set_xticklabels(etiquetas)
ax.set_ylabel('Tasa de Error')
ax.set_title('CER y WER por Documento — EasyOCR (Barrera 3)', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('cer_wer_barrera3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafica guardada como: cer_wer_barrera3.png')

   METRICAS OCR — BARRERA 3 (EasyOCR)
  CER Global (Character Error Rate): 0.0311  (3.11 porciento)
  WER Global (Word Error Rate):      0.2500  (25.00 porciento)

  Interpretacion:
    De cada 100 caracteres, el OCR comete 3.1 errores
    De cada 100 palabras,   el OCR comete 25.0 errores

  RESULTADO: CER dentro del estandar profesional (menos de 5 porciento)
Grafica guardada como: cer_wer_barrera3.png


In [14]:
# ============================================================= #
# RESUMEN FINAL — TABLA DE METRICAS COMPLETA DEL SISTEMA        #
# ============================================================= #

print('=' * 58)
print('   RESUMEN DE METRICAS CIENTIFICAS DEL SISTEMA')
print('   Sistema de Validacion Documental con IA')
print('   Sprint 9 — Agosto 2026')
print('=' * 58)
print()
print('BARRERA 1 — Clasificador CNN (MobileNetV3-Small)')
print(f'  Exactitud  (Accuracy): {acc * 100:.2f} porciento')
print(f'  Precision  (Macro):    {prec * 100:.2f} porciento')
print(f'  Recall     (Macro):    {rec * 100:.2f} porciento')
print(f'  F1-Score   (Macro):    {f1 * 100:.2f} porciento')
print(f'  AUC-ROC:               {roc_auc:.4f}')
print()
print('BARRERA 3 — OCR (EasyOCR + Regex RENAPO)')
print(f'  CER (Character Error Rate): {cer_total * 100:.2f} porciento')
print(f'  WER (Word Error Rate):      {wer_total * 100:.2f} porciento')
print()
print('Graficas generadas:')
print('  > matriz_confusion_barrera1.png')
print('  > curva_roc_barrera1.png')
print('  > cer_wer_barrera3.png')
print()
print('Referencias:')
print('  Noguti et al. (2020). arXiv:2010.12533')
print('  Nagaonkar et al. (2025). arXiv:2502.06445')
print('  Saifullah et al. (2024). arXiv:2407.03830')

   RESUMEN DE METRICAS CIENTIFICAS DEL SISTEMA
   Sistema de Validacion Documental con IA
   Sprint 9 — Agosto 2026

BARRERA 1 — Clasificador CNN (MobileNetV3-Small)


NameError: name 'acc' is not defined